In [1]:
import os
os.chdir("/home/jovyan/work/repo_conteo_gallinas")
os.environ["CUDA_VISIBLE_DEVICES"] = "0, 2"

import torch
from ultralytics import YOLO
import pandas as pd
import shutil
import gc

import optuna


/home/jovyan/work/repo_conteo_gallinas/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def objective(trial):

    model = YOLO("model/yolo26m.pt")

    # Eliminar caarpetas de resultados
    if os.path.exists("runs"):
        shutil.rmtree("runs")
    
    params = {
        "hsv_s": trial.suggest_float("hsv_s", 0.0, 1.0),
        "hsv_v": trial.suggest_float("hsv_v", 0.0, 1.0),
        "degrees": trial.suggest_float("degrees", 0.0, 180.0),
        "translate": trial.suggest_float("translate", 0.0, 1.0),
        "scale": trial.suggest_float("scale", 0.0, 1.0),
        "perspective": trial.suggest_float("perspective", 0.0, 0.001),
        "fliplr": trial.suggest_float("fliplr", 0.0, 1.0),
        "flipud": trial.suggest_float("flipud", 0.0, 1.0),
        "erasing": trial.suggest_float("erasing", 0.0, 1.0)
    }

    results = model.train(
        data="artifacts/full_dataset.yaml",
        imgsz=640,
        epochs=291,
        batch=11,
        lr0=0.00215269748298882,
        optimizer="SGD",
        seed=123,
        deterministic=True,
        
        device=[0, 1],
        workers=0,
        project='.',
        name='.',

        # Data aumentation
        hsv_s=params["hsv_s"],
        hsv_v=params["hsv_v"],
        degrees=params["degrees"],
        translate=params["translate"],
        scale=params["scale"],
        perspective=params["perspective"],
        fliplr=params["fliplr"],
        flipud=params["flipud"],
        erasing=params["erasing"]
    )
        
    # Lectura resultados
    df = pd.read_csv("runs/detect/results.csv")
    
    best = df.loc[df["metrics/mAP50-95(B)"].idxmax()]
    map5095_final = float(best["metrics/mAP50-95(B)"])

    del model
    gc.collect()
    torch.cuda.empty_cache()

    return map5095_final


In [ ]:
study = optuna.create_study(
    direction="maximize",
    study_name="yolo_detect",
    sampler=optuna.samplers.TPESampler(seed=123)
)

study.optimize(
    objective,
    n_trials=15
)

df_results = study.trials_dataframe()
df_results.to_csv(
    "tmp/yolo26_data_augmentation_11.csv",
    index=False
)



[I 2026-11-11 21:10:55,282] A new study created in memory with name: yolo_detect


Ultralytics 8.4.96 🚀 Python-3.13.11 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX 6000 Ada Generation, 48640MiB)
                                                       CUDA:1 (NVIDIA RTX 6000 Ada Generation, 48632MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=11, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=artifacts/full_dataset.yaml, degrees=40.83326164155656, deterministic=True, device=0,1, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=291, erasing=0.48093190148436094, exist_ok=False, fliplr=0.9807641983846155, flipud=0.6848297385848633, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.6964691855978616, hsv_v=0.28613933495037946, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0

[I 2026-11-11 21:25:39,843] Trial 0 finished with value: 0.19874 and parameters: {'hsv_s': 0.6964691855978616, 'hsv_v': 0.28613933495037946, 'degrees': 40.83326164155656, 'translate': 0.5513147690828912, 'scale': 0.7194689697855631, 'perspective': 0.00042310646012446096, 'fliplr': 0.9807641983846155, 'flipud': 0.6848297385848633, 'erasing': 0.48093190148436094}. Best is trial 0 with value: 0.19874.


Ultralytics 8.4.96 🚀 Python-3.13.11 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX 6000 Ada Generation, 48640MiB)
                                                       CUDA:1 (NVIDIA RTX 6000 Ada Generation, 48632MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=11, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=artifacts/full_dataset.yaml, degrees=131.2289473291275, deterministic=True, device=0,1, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=291, erasing=0.17545175614749253, exist_ok=False, fliplr=0.7379954057320357, flipud=0.18249173045349998, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.3921175181941505, hsv_v=0.3431780161508694, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0

[I 2026-11-11 21:40:18,960] Trial 1 finished with value: 0.2344 and parameters: {'hsv_s': 0.3921175181941505, 'hsv_v': 0.3431780161508694, 'degrees': 131.2289473291275, 'translate': 0.4385722446796244, 'scale': 0.05967789660956835, 'perspective': 0.00039804425533043145, 'fliplr': 0.7379954057320357, 'flipud': 0.18249173045349998, 'erasing': 0.17545175614749253}. Best is trial 1 with value: 0.2344.


Ultralytics 8.4.96 🚀 Python-3.13.11 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX 6000 Ada Generation, 48640MiB)
                                                       CUDA:1 (NVIDIA RTX 6000 Ada Generation, 48632MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=11, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=artifacts/full_dataset.yaml, degrees=114.19217253923779, deterministic=True, device=0,1, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=291, erasing=0.3617886556223141, exist_ok=False, fliplr=0.7224433825702216, flipud=0.3229589138531782, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.5315513738418384, hsv_v=0.5318275870968661, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=

... (16 outputs left out)

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 7.8it/s 0.3s0.4s
                   all         38       3851      0.645      0.656      0.635      0.218

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
    198/291      5.12G      1.943      1.047   0.005718        633        640: 100% ━━━━━━━━━━━━ 15/15 6.8it/s 2.2s0.1s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 7.6it/s 0.3s0.4s
                   all         38       3851       0.64      0.668       0.64      0.222

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
    199/291      5.12G      1.942      1.044   0.005889        807        640: 100% ━━━━━━━━━━━━ 15/15 7.0it/s 2.2s0.1s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 7.8it/s 0.3s0.4s
                   all         38    

[I 2026-11-12 00:01:24,400] Trial 11 finished with value: 0.24971 and parameters: {'hsv_s': 0.9987800535744109, 'hsv_v': 0.10491521663182801, 'degrees': 2.3034710470253543, 'translate': 0.20579142456083427, 'scale': 0.22724854922669604, 'perspective': 0.0009967816612219058, 'fliplr': 0.48791848389248094, 'flipud': 0.0153252900803037, 'erasing': 0.9877460034715213}. Best is trial 10 with value: 0.25445.


Ultralytics 8.4.96 🚀 Python-3.13.11 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX 6000 Ada Generation, 48640MiB)
                                                       CUDA:1 (NVIDIA RTX 6000 Ada Generation, 48632MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=11, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=artifacts/full_dataset.yaml, degrees=36.73547556157239, deterministic=True, device=0,1, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=291, erasing=0.758538980808829, exist_ok=False, fliplr=0.36001257638319506, flipud=0.9704401202027961, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.9726105484687346, hsv_v=0.016077304413945448, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0

In [ ]:
df_results